# Credit Card Fraud Detection USING ML

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,precision_recall_curve,auc

from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

In [ ]:
df = pd.read_csv("/content/creditcard_5000.csv")
df.head()

In [ ]:
print(df.shape)
print(df["Class"].value_counts())


plt.figure(figsize=(5,4))
sns.countplot(data=df, x="Class")
plt.title("Fraud(1) vs Non-Fraud(0)")
plt.show()


#Split Features & Labels


In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

# Train-test split


In [ ]:
X_train, X_test, y_train, y_test =train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)

 # Standardize Amount & Time


In [ ]:
scaler = StandardScaler()
X_train[['Amount','Time']] = scaler.fit_transform(X_train[['Amount','Time']])
X_test[['Amount','Time']] = scaler.transform(X_test[['Amount','Time']])

#Handle Imbalance with SMOTE

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res =sm.fit_resample(X_train, y_train)

In [ ]:
print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_res.value_counts())

# Train XGBoost Fraud Detector

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

model.fit(X_train_res, y_train_res)

 # Predictions & Performance


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred))


# confusion Matrix

In [ ]:
cm =confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("confusion Matrix")
plt.show()

# PR curve +PR AUC

In [ ]:
# Precision-Recall AUC (very important for fraud)
precision,recall,thresholds =precision_recall_curve(y_test,y_proba)
pr_auc = auc(recall,precision)
print("PR-AUC:", pr_auc)

plt.plot(recall,precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("PR curve")
plt.show()

# ROC-AUC

In [ ]:
rocauc =roc_auc_score(y_test,y_proba)
rocauc

# SHAP Explainability (Why a Transaction Is Fraud)


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)

#  Try New Transaction (Manual Input)

In [ ]:
sample = X_test.iloc[20:21]   # pick any row or make your own
proba = model.predict_proba(sample)[0][1]
proba
